# Review Anomalies

Step through unreviewed quality findings one by one. Inspect the SimFin numbers, open the SEC filing for verification, annotate the reason, mark reviewed.

Reviews are stored in `data/data_quality/anomaly_reviews.toml`. Future runs of `run()` skip already-reviewed (ticker, period, rule) keys.

In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

from irp.data import statement
from irp.quality import run, inspect, add_review, add_flag, load_reviews_df, load_flags_df

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)

SAMPLE = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'JPM', 'XOM', 'JNJ', 'WMT']
VARIANT = 'A'

## Load Queue

In [ ]:
queue = run(SAMPLE, VARIANT, skip_reviewed=True)
print(f'{len(queue)} unreviewed findings')
if len(queue):
    display(queue[['Ticker', 'Period_str', 'Rule', 'rel_diff']].head(20).style.format({'rel_diff': '{:.2%}'}))

## Reviewer

Single-finding loop. `Mark Reviewed & Next` appends to the TOML and advances. `Skip` advances without saving. Re-run `Load Queue` cell to refresh.

In [3]:
state = {'idx': 0}

note_w = widgets.Textarea(
    description='Note:',
    layout=widgets.Layout(width='800px', height='80px'),
)
status_w = widgets.Dropdown(
    options=['ok', 'data_error', 'to_check'],
    description='Status:',
    value='ok',
)
next_btn = widgets.Button(description='Mark Reviewed & Next', button_style='primary')
skip_btn = widgets.Button(description='Skip')
output = widgets.Output()


def _fmt_num(x):
    if pd.isna(x):
        return ''
    try:
        return f'{float(x):,.0f}'
    except (TypeError, ValueError):
        return str(x)


def render():
    with output:
        clear_output()
        if state['idx'] >= len(queue):
            print('Queue empty. Re-run the Load Queue cell to refresh.')
            return
        row = queue.iloc[state['idx']]
        progress = f'{state["idx"] + 1}/{len(queue)}'
        link = f'<a href="{row["EDGAR"]}" target="_blank">open EDGAR</a>' if row.get('EDGAR') else '(no EDGAR url)'
        display(HTML(
            f'<h3>[{progress}] {row["Ticker"]} {row["Period_str"]} — {row["Rule"]}</h3>'
            f'<p><b>{row["LHS_value"]:,.0f}</b> vs <b>{row["RHS_value"]:,.0f}</b> '
            f'(diff {row["diff"]:,.0f}, {row["rel_diff"]:.2%}) &nbsp; | &nbsp; {link}</p>'
        ))
        ins = inspect(
            row['Rule'], row['Ticker'],
            int(row['Fiscal Year']), str(row['Fiscal Period']), str(row['Period']),
        )
        if len(ins):
            num_cols = [c for c in ins.columns if ins[c].dtype.kind in 'fi']
            display(ins.style.format({c: _fmt_num for c in num_cols}))
        note_w.value = ''


def on_next(_):
    if state['idx'] >= len(queue):
        return
    row = queue.iloc[state['idx']]
    add_review(row['Ticker'], row['Period_str'], row['Rule'], status_w.value, note_w.value)
    state['idx'] += 1
    render()


def on_skip(_):
    state['idx'] += 1
    render()


next_btn.on_click(on_next)
skip_btn.on_click(on_skip)

display(widgets.VBox([
    output,
    note_w,
    status_w,
    widgets.HBox([next_btn, skip_btn]),
]))
render()

## Side-Findings

Flag a SimFin issue **not** caught by a rule. `Subject` is free-form scope — a single field, a full statement, or anything in between (e.g. `Revenue`, `Cash Flow Statement`, `Equity bridge`). Ticker + period prefill from the current review item; override if needed.

In [4]:
_flag_ticker_w = widgets.Text(description='Ticker:', value='')
_flag_period_w = widgets.Text(description='Period:', placeholder='2024FY or 2024Q2')
_flag_subject_w = widgets.Text(
    description='Subject:',
    placeholder='Revenue, Cash Flow Statement, Equity bridge, …',
    layout=widgets.Layout(width='600px'),
)
_flag_status_w = widgets.Dropdown(options=['ok', 'data_error', 'to_check'], description='Status:', value='to_check')
_flag_note_w = widgets.Textarea(description='Note:', layout=widgets.Layout(width='800px', height='60px'))
_flag_btn = widgets.Button(description='Add Flag', button_style='warning')
_flag_out = widgets.Output()


def _prefill_from_current():
    if state['idx'] < len(queue):
        row = queue.iloc[state['idx']]
        _flag_ticker_w.value = str(row['Ticker'])
        _flag_period_w.value = str(row['Period_str'])


def _on_add_flag(_):
    t = _flag_ticker_w.value.strip()
    p = _flag_period_w.value.strip()
    s = _flag_subject_w.value.strip()
    n = _flag_note_w.value.strip()
    with _flag_out:
        clear_output()
        if not (t and p and s):
            print('Need ticker, period, and subject.')
            return
        add_flag(t, p, s, _flag_status_w.value, n)
        print(f'Flagged: {t} {p} subject={s!r} status={_flag_status_w.value}')
        _flag_subject_w.value = ''
        _flag_note_w.value = ''


_flag_btn.on_click(_on_add_flag)
_prefill_from_current()

display(widgets.VBox([
    widgets.HBox([_flag_ticker_w, _flag_period_w]),
    _flag_subject_w,
    _flag_note_w,
    _flag_status_w,
    _flag_btn,
    _flag_out,
]))

## Statement Viewer

Pull income / balance / cashflow for a ticker across one or more periods. Periods comma-separated, e.g. `2024FY, 2023FY, 2024Q2`. Leave blank for all available.

In [ ]:
_view_ticker_w = widgets.Text(description='Ticker:', value='')
_view_stmt_w = widgets.Dropdown(options=['income', 'balance', 'cashflow'], description='Statement:', value='income')
_view_periods_w = widgets.Text(
    description='Periods:',
    placeholder='2024FY, 2023FY, 2024Q2  (blank = all)',
    layout=widgets.Layout(width='600px'),
)
_view_items_w = widgets.Text(
    description='Items:',
    placeholder='Revenue, Gross Profit  (blank = all)',
    layout=widgets.Layout(width='600px'),
)
_view_btn = widgets.Button(description='Show', button_style='info')
_view_out = widgets.Output()


def _prefill_viewer():
    if state['idx'] < len(queue):
        row = queue.iloc[state['idx']]
        _view_ticker_w.value = str(row['Ticker'])
        _view_periods_w.value = str(row['Period_str'])


def _on_view(_):
    t = _view_ticker_w.value.strip()
    periods_s = _view_periods_w.value.strip()
    items_s = _view_items_w.value.strip()
    periods = [p.strip() for p in periods_s.split(',') if p.strip()] or None
    items = [p.strip() for p in items_s.split(',') if p.strip()] or None
    with _view_out:
        clear_output()
        if not t:
            print('Need ticker.')
            return
        df = statement(t, _view_stmt_w.value, periods=periods, items=items)
        if df.empty:
            print('No data for that ticker/periods.')
            return
        display(df.style.format(_fmt_num))


_view_btn.on_click(_on_view)
_prefill_viewer()

display(widgets.VBox([
    widgets.HBox([_view_ticker_w, _view_stmt_w]),
    _view_periods_w,
    _view_items_w,
    _view_btn,
    _view_out,
]))

## Audit — All Reviews

In [6]:
_reviews = load_reviews_df()
print(f'{len(_reviews)} total entries (rule-reviews + manual flags)')
display(_reviews)

print()
_flags = load_flags_df()
print(f'{len(_flags)} manual flags')
display(_flags)

10 total entries (rule-reviews + manual flags)


,ticker,period,rule,status,note,reviewed_at,subject
0,AMZN,2024FY,cash_chain,ok,"Missing Foreign currency effect on cash, cash ...",2026-05-17,NaN
1,AMZN,2023FY,cash_chain,ok,"Missing Foreign currency effect on cash, cash ...",2026-05-17,NaN
2,AMZN,2022FY,cash_chain,ok,"Missing Foreign currency effect on cash, cash ...",2026-05-17,NaN
3,AMZN,2021FY,cash_chain,ok,"Missing Foreign currency effect on cash, cash ...",2026-05-17,NaN
4,AMZN,2020FY,cash_chain,ok,"Missing Foreign currency effect on cash, cash ...",2026-05-17,NaN
5,JNJ,2024FY,cash_chain,ok,Missing Effect of exchange rate changes on cas...,2026-05-17,NaN
6,JNJ,2023FY,cash_chain,ok,Missing Effect of exchange rate changes on cas...,2026-05-17,NaN
7,JNJ,2021FY,cash_chain,ok,Missing Effect of exchange rate changes on cas...,2026-05-17,NaN
8,JNJ,2020FY,cash_chain,ok,Missing Effect of exchange rate changes on cas...,2026-05-17,NaN
9,META,2022FY,manual,to_check,,2026-05-17,Cashflow



1 manual flags


,ticker,period,rule,status,note,reviewed_at,subject
0,META,2022FY,manual,to_check,,2026-05-17,Cashflow


In [7]:
_reviews['note'].values

<ArrowStringArray>
['Missing Foreign currency effect on cash, cash equivalents, and restricted cash -1,301,000,000',    'Missing Foreign currency effect on cash, cash equivalents, and restricted cash 403,000,000',
 'Missing Foreign currency effect on cash, cash equivalents, and restricted cash -1,093,000,000',   'Missing Foreign currency effect on cash, cash equivalents, and restricted cash -364,000,000',
   'Missing Foreign currency effect on cash, cash equivalents, and restricted cash 618,000,000,',             'Missing Effect of exchange rate changes on cash and cash equivalents -289,000,000',
             'Missing Effect of exchange rate changes on cash and cash equivalents -112,000,000',             'Missing Effect of exchange rate changes on cash and cash equivalents -178,000,000',
              'Missing Effect of exchange rate changes on cash and cash equivalents -89,000,000',                                                                                              '']
Length